In [19]:
import numpy as np
import pandas as pd
from pydantic import BaseModel, Field
from typing import List, Tuple, Literal

import SoD_Utils
import Text_Utils

In [2]:
LANGUAGE = SoD_Utils.LANGUAGE_CZ

data_path = path = SoD_Utils.get_dataset_path(LANGUAGE)
df = pd.read_spss(data_path)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3880 entries, 0 to 3879
Columns: 170 entries, id to lca_DK_n_8
dtypes: category(163), float64(7)
memory usage: 864.0 KB


In [3]:
def process_SoD_response(row):
    # --- 1. Basic Information ---
    # Renames Czech keys to English and performs initial data cleaning.
    processed_data = {
        'gender': row.get('GENDER').lower(),
        'age': int(row.get('AGE1', 0)),
        'education_level': row.get('EDU', '').lower(),
        'region': row.get('KRAJ'),
        'district': row.get('OKRES'),
        'town_size': SoD_Utils.process_town_size(row),
        'employment_status': SoD_Utils.process_employment(row),
        'income_range': SoD_Utils.process_income(row),
    }

    # --- 2. Additional Survey Questions ---
    # Merges the dictionary of additional questions into the main one.
    additional_data = {
        'living_standard': row.get('Q19','').lower(),
        'interest_in_politics': row.get('Q20','').lower(),
        'opinion_on_eu': row.get('Q18','').lower(),
        'opinion_on_nato': row.get('Q17','').lower(),
        'covid_vaccinated': row.get('Q23','').lower(),
    }
    processed_data.update(additional_data)


    return processed_data

data = [process_SoD_response(df.loc[i]) for i in df.index]
data = pd.DataFrame(data, index=df.index)

In [4]:
data

,gender,age,education_level,region,district,town_size,employment_status,income_range,living_standard,interest_in_politics,opinion_on_eu,opinion_on_nato,covid_vaccinated
0,muž,28,vysokoškolské vzdělání,Středočeský kraj,Nymburk,Méně než 1.000 obyvatel,zaměstnanec na plný úvazek,30.001 - 40.000 Kč,spíše dobrou,velmi se zajímám,spíše spokojený/á,rozhodně spokojený/á,ano
1,muž,27,vysokoškolské vzdělání,Plzeňský kraj,Plzeň-město,Více než 100.000 obyvatel,zaměstnanec na plný úvazek,20.001 - 25.000 Kč,spíše dobrou,spíše se zajímám,spíše spokojený/á,spíše spokojený/á,ano
2,žena,33,základní + středoškolské vzdělání bez maturity,Královéhradecký kraj,Jičín,Méně než 1.000 obyvatel,zaměstnanec na plný úvazek,25.001 - 30.000 Kč,"ani dobrou, ani špatnou",spíše se zajímám,spíše spokojený/á,rozhodně spokojený/á,ano
3,žena,27,základní + středoškolské vzdělání bez maturity,Plzeňský kraj,Rokycany,20.001 - 100.000 obyvatel,zaměstnanec na plný úvazek,20.001 - 25.000 Kč,spíše dobrou,spíše se zajímám,rozhodně spokojený/á,rozhodně spokojený/á,ano
4,muž,21,středoškolské vzdělání s maturitou,Ústecký kraj,Chomutov,20.001 - 100.000 obyvatel,zaměstnanec na plný úvazek,30.001 - 40.000 Kč,velmi dobrou,vůbec se nezajímám,rozhodně spokojený/á,rozhodně spokojený/á,ano
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3875,žena,39,středoškolské vzdělání s maturitou,Plzeňský kraj,Plzeň-město,Více než 100.000 obyvatel,zaměstnanec na plný úvazek,40.001 - 60.000 Kč,spíše dobrou,spíše se nezajímám,spíše spokojený/á,spíše spokojený/á,ne
3876,žena,30,vysokoškolské vzdělání,Hlavní město Praha,Praha,Více než 100.000 obyvatel,zaměstnanec na částečný úvazek,None,spíše dobrou,spíše se zajímám,rozhodně spokojený/á,rozhodně spokojený/á,ano
3877,žena,47,vysokoškolské vzdělání,Moravskoslezský kraj,Karviná,5.001 - 20.000 obyvatel,zaměstnanec na plný úvazek,40.001 - 60.000 Kč,spíše dobrou,vůbec se nezajímám,spíše spokojený/á,rozhodně spokojený/á,ano
3878,žena,23,středoškolské vzdělání s maturitou,Hlavní město Praha,Praha,20.001 - 100.000 obyvatel,zaměstnanec na plný úvazek,20.001 - 25.000 Kč,spíše špatnou,nevím,nevím,nevím,nechci uvést


In [13]:
def _format_opinion_statement(gender, opinion, topic_string):
    """
    Creates a full opinion sentence with correct gendered adjectives.
    Example: (gender='Muž', opinion='Rozhodně ano', topic='EU')
             -> "Jsem rozhodně přesvědčený, že je Česká republika členem EU."

    This helper removes code duplication for the EU and NATO questions.
    """
    if opinion == 'Nevím':
        return "" # Return an empty string if there is no opinion

    return f"Jsem {Text_Utils.declension_gender_ya(opinion[:-3],gender).lower()}, že je Česká republika členským státem {topic_string}."
# --- Main Function to Create the Description ---

def create_respondent_description(respondent):
    """
    Generates a descriptive Czech paragraph about a survey respondent
    by combining their answers into grammatically correct sentences.

    Args:
        respondent (dict): A dictionary containing the processed data for one person,
                           with English keys (e.g., 'gender', 'region').

    Returns:
        str: A multi-sentence description of the respondent in Czech.
    """
    # --- 1. Build the description sentence by sentence ---
    # Using a list of sentences is cleaner than repeated string concatenation.
    gender = SoD_Utils.gender_to_enum_gender(respondent['gender'])
    description_parts = []

    # --- Basic Demographics ---
    description_parts.append(
        f"Jsem {respondent['gender']}, "
        f"je mi {respondent['age']} let, "
        f"mé vzdělání je {respondent['education_level']}."
    )

    # --- Location ---
    # This now uses the helper function for complex Czech grammar.
    description_parts.append(
        f"Žiji v {SoD_Utils.decline_region_to_Locative(respondent['region'])}, "
        f"v okresu {respondent['district']} a "
        f"obci o velikosti {respondent['town_size']}."
    )

    # --- Socioeconomic Status ---
    description_parts.append(f"Z hlediska zaměstnání jsem {respondent['employment_status']}")
    if respondent['income_range']:
        description_parts.append(f"a příjem naší domácnosti je {respondent['income_range']}")
    description_parts[-1]+="."

    # EU and NATO opinions now use the dedicated helper function
    description_parts.append(_format_opinion_statement(gender, respondent['opinion_on_eu'], "EU"))
    description_parts.append(_format_opinion_statement(gender, respondent['opinion_on_nato'], "NATO"))

    # Living Standard
    living_standard = respondent['living_standard']
    if SoD_Utils.is_non_substantive_responses(living_standard):
        verb = 'mám'
        if 'ani' in respondent['living_standard']:
            verb = Text_Utils.declension_negation_ne(verb)
        description_parts.append(f"{verb.capitalize()} {living_standard} životní úroveň.")

    # Interest in Politics
    interest = respondent['interest_in_politics']
    if SoD_Utils.is_non_substantive_responses(interest):
        description_parts.append(f"{interest.capitalize()} o politiku.")

    # COVID Vaccination Status
    vacc_status = respondent['covid_vaccinated']
    if SoD_Utils.is_non_substantive_responses(vacc_status):
        is_vaccinated = Text_Utils.parse_yes_no(vacc_status)
        verb = 'jsem'
        if not is_vaccinated:
            verb = Text_Utils.declension_negation_ne(verb)
        description_parts.append(f"{verb.capitalize()} {Text_Utils.declension_gender_ya('očkován',gender)} proti covidu.")

    # --- 2. Combine all parts into a final paragraph ---
    # Filter out any empty strings that may have been returned by helpers (e.g., for 'Nevím' answers)
    # and join the parts with a space.
    full_description = " ".join(part for part in description_parts if part)

    return full_description

In [15]:
create_respondent_description(data.iloc[2])

'Jsem žena, je mi 33 let, mé vzdělání je základní + středoškolské vzdělání bez maturity. Žiji v Královéhradeckém kraji, v okresu Jičín a obci o velikosti Méně než 1.000 obyvatel. Z hlediska zaměstnání jsem zaměstnanec na plný úvazek a příjem naší domácnosti je 25.001 - 30.000 Kč. Jsem spíše spokojená, že je Česká republika členským státem EU. Jsem rozhodně spokojená, že je Česká republika členským státem NATO.'

## Calling prompts

In [7]:
czech_prompt_start = """Po značce [INSERT] doplň, zda respondent volil ve volbách do poslanecké sněmovny 2021 a pokud ano, pro jakou stranu hlasoval.
Možné strany, kterým nejspíše dal hlas, vypiš s pravděpodobností že pro danou stranu hlasoval. Vypiš kolik stran je potřeba.
Případně můžeš vypsat i "jiná strana", což představuje hlas pro málo populární stranu. Součet pravděpodobností musí být 1.0.

Formát výstupu je:
*[volil, proba a], [nevolil, proba b]; [PARTY1, proba 1], [PARTY2, proba 2],... , [PARTYN, proba n]*
 kde PARTY1, PARTY2,... jsou jména stram, proba a, proba b, proba 1,... je třeba nahradit desetinnými čísly (pravděpodobnost). Za druhý znak "*" už nic nevypisuj.

 Zachovej správné oddělovače (';' mezi sekcí volil/nevolil a sekcí stran, všude jinde ','). Volil/nevolil je první, pak následují strany.
"""

czech_prompt_end = " Ve volbách do poslanecké sněmovny 2021 jsem [INSERT]"

In [8]:
ex_prompt = czech_prompt_start + create_respondent_description(data.iloc[0]) + czech_prompt_end
ex_prompt

'Po značce [INSERT] doplň, zda respondent volil ve volbách do poslanecké sněmovny 2021 a pokud ano, pro jakou stranu hlasoval.\nMožné strany, kterým nejspíše dal hlas, vypiš s pravděpodobností že pro danou stranu hlasoval. Vypiš kolik stran je potřeba.\nPřípadně můžeš vypsat i "jiná strana", což představuje hlas pro málo populární stranu. Součet pravděpodobností musí být 1.0.\n\nFormát výstupu je:\n*[volil, proba a], [nevolil, proba b]; [PARTY1, proba 1], [PARTY2, proba 2],... , [PARTYN, proba n]*\n kde PARTY1, PARTY2,... jsou jména stram, proba a, proba b, proba 1,... je třeba nahradit desetinnými čísly (pravděpodobnost). Za druhý znak "*" už nic nevypisuj.\n\n Zachovej správné oddělovače (\';\' mezi sekcí volil/nevolil a sekcí stran, všude jinde \',\'). Volil/nevolil je první, pak následují strany.\nJsem muž, je mi 28 let,  mé vzdělání je vysokoškolské vzdělání. Žiji v Středočeském kraji, v okresu Nymburk a obci o velikosti Méně než 1.000 obyvatel.  Z hlediska zaměstnání jsem zaměstn

In [9]:
#TODO: GPT 4 nano
# pydantic structured data
# poslat vysledky
# ciel : structured output works

In [20]:
class PartyProbability(BaseModel):
    """
    Strukturovaná reprezentace pravděpodobnosti hlasování pro konkrétní stranu.
    """
    name: Literal[
        "ANO 2011",
        "Koalice Spolu (ODS, TOP 09, KDU-ČSL)",
        "Koalice PIRÁTI a STAROSTOVÉ",
        "Komunistická strana Čech a Moravy (KSČM)",
        "Svoboda a přímá demokracie – Tomio Okamura (SPD)",
        "Česká strana sociálně demokratická (ČSSD)",
        "Trikolóra, Svobodní a Soukromníci",
        "Přísaha Roberta Šlachty",
        "Jiná strana"
    ] = Field(description="Název politické strany.")
    probability: float = Field(
        ge=0, le=1, description="Pravděpodobnost, že respondent hlasoval pro tuto stranu."
    )

class VotingResult(BaseModel):
    """
    Strukturovaný výstup pro volební chování respondenta ve volbách do poslanecké sněmovny 2021.
    """
    voted_or_not: Tuple[
        Tuple[Literal["volil"], float],
        Tuple[Literal["nevolil"], float]
    ] = Field(
        description="Dvojice, obsahující dvě dvojice, reprezentující pravděpodobnost hlasování a nehlasování. Součet pravděpodobností musí být 1.0."
    )
    parties: List[PartyProbability] = Field(
        description="Seznam možných stran, pro které respondent hlasoval, spolu s pravděpodobností pro každou z nich. Součet všech pravděpodobností musí být 1.0."
    )